# **SQL Queries**

**1. DATA LOADING**

In [27]:
import pandas as pd
import sqlite3
import re

file_path = '/content/anime_catalog_clean.csv'
raw_df = pd.read_csv(file_path)

**2. DATABASE SETUP (SQLite in-memory)**

Creating tables according to relational schema (Many-to-Many logic)

In [28]:
import os

db_dir = 'database/db'
if not os.path.exists(db_dir):
    os.makedirs(db_dir)

conn = sqlite3.connect(os.path.join(db_dir, 'anime_catalog.db'))
cursor = conn.cursor()

cursor.executescript("""
-- Drop tables if they exist to allow recreation with new schema
DROP TABLE IF EXISTS anime_genres;
DROP TABLE IF EXISTS anime_studios;
DROP TABLE IF EXISTS anime_directors;
DROP TABLE IF EXISTS anime_dubbing_groups; -- New junction table
DROP TABLE IF EXISTS anime_stats;
DROP TABLE IF EXISTS anime;
DROP TABLE IF EXISTS genres;
DROP TABLE IF EXISTS studios;
DROP TABLE IF EXISTS directors;
DROP TABLE IF EXISTS dubbing_groups;

-- 1. Reference Tables (Dictionaries)
CREATE TABLE genres (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);
CREATE TABLE studios (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);
CREATE TABLE directors (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);
CREATE TABLE dubbing_groups (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);

-- 2. Main Anime Table
CREATE TABLE anime (
    anime_id INT PRIMARY KEY,
    title_ru TEXT,
    alt_names TEXT,
    anime_type TEXT,
    status TEXT,
    release_year INT,
    age_rating TEXT,
    source TEXT,
    url TEXT
);

-- 3. Statistics Table (1:1 Relationship with Anime)
CREATE TABLE anime_stats (
    anime_id INT PRIMARY KEY REFERENCES anime(anime_id) ON DELETE CASCADE,
    site_rating DECIMAL,
    site_votes INT,
    site_views INT
);

-- 4. Junction Tables (Many-to-Many Relationships)
CREATE TABLE anime_genres (
    anime_id INT REFERENCES anime(anime_id),
    genre_id INT REFERENCES genres(id),
    PRIMARY KEY (anime_id, genre_id)
);
CREATE TABLE anime_studios (
    anime_id INT REFERENCES anime(anime_id),
    studio_id INT REFERENCES studios(id),
    PRIMARY KEY (anime_id, studio_id)
);
CREATE TABLE anime_directors (
    anime_id INT REFERENCES anime(anime_id),
    director_id INT REFERENCES directors(id),
    PRIMARY KEY (anime_id, director_id)
);
CREATE TABLE anime_dubbing_groups ( -- New junction table
    anime_id INT REFERENCES anime(anime_id),
    dubbing_group_id INT REFERENCES dubbing_groups(id),
    PRIMARY KEY (anime_id, dubbing_group_id)
);
""")

**3. CLEANING FUNCTIONS & MIGRATION**

In [29]:
def to_int(val):
    if pd.isna(val) or val == 'N/A': return 0
    num = re.sub(r'\D', '', str(val))
    return int(num) if num else 0

def to_float(val):
    if pd.isna(val) or val == 'N/A': return 0.0
    try: return float(val)
    except: return 0.0

def link_entity(anime_id, column_val, table_name, link_table, link_col_name):
    """Helper function to populate dictionaries and link tables"""
    if pd.isna(column_val) or column_val == 'N/A': return

    entities = [e.strip() for e in str(column_val).split('|')]
    for name in entities:
        cursor.execute(f"INSERT OR IGNORE INTO {table_name} (name) VALUES (?)", (name,))
        entity_id = cursor.execute(f"SELECT id FROM {table_name} WHERE name = ?", (name,)).fetchone()[0]
        cursor.execute(f"INSERT OR IGNORE INTO {link_table} (anime_id, {link_col_name}) VALUES (?, ?)", (anime_id, entity_id))

for _, row in raw_df.iterrows():

    year_match = re.search(r'(\d{4})', str(row['year']))
    year = int(year_match.group(1)) if year_match else None

    cursor.execute("""
        INSERT OR IGNORE INTO anime (anime_id, title_ru, alt_names, anime_type, status, release_year, age_rating, source, url)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (row['anime_id'], row['title_ru'], row['alt_names'], row['type'], row['status'], year, row['age_rating'], row['source'], row['url']))

    cursor.execute("""
        INSERT OR IGNORE INTO anime_stats (anime_id, site_rating, site_votes, site_views)
        VALUES (?, ?, ?, ?)
    """, (row['anime_id'], to_float(row['site_rating']), row['site_votes'], to_int(row['site_views']))) # Removed non-existent columns

    link_entity(row['anime_id'], row['genres'], 'genres', 'anime_genres', 'genre_id')
    link_entity(row['anime_id'], row['studio'], 'studios', 'anime_studios', 'studio_id')
    link_entity(row['anime_id'], row['director'], 'directors', 'anime_directors', 'director_id')
    link_entity(row['anime_id'], row['dubbing'], 'dubbing_groups', 'anime_dubbing_groups', 'dubbing_group_id') # Added for dubbing

conn.commit()
print("Success: Data migrated to relational structure.")

Success: Data migrated to relational structure.


**4. DATA ANALYSIS (SQL QUERIES)**

In [30]:
def run_query(title, description, sql):
    print(f"--- Query: {title} ---")
    print(f"Insight: {description}")
    display(pd.read_sql(sql, conn))
    print("\n")

Query 1: COUNT + GROUP BY + ORDER BY

In [31]:
run_query(
    "Distribution by Format",
    "Counts the number of anime in each category (ONA, TV, Movie) to understand catalog variety.",
    "SELECT anime_type, COUNT(*) as total FROM anime GROUP BY anime_type ORDER BY total DESC"
)

--- Query: Distribution by Format ---
Insight: Counts the number of anime in each category (ONA, TV, Movie) to understand catalog variety.


,anime_type,total
0,Сериал,4142
1,ONA,1727
2,OVA,1353
3,Полнометражный фильм,1318
4,Спешл,817
5,Малометражный сериал,373
6,Короткометражный фильм,305
7,Неизвестно,107


 Query 2: AVG + JOIN + GROUP BY

In [32]:
run_query(
    "Average Rating by Age Category",
    "Calculates the average user rating for different age ratings to see target audience satisfaction.",
    """
    SELECT a.age_rating, ROUND(AVG(s.site_rating), 2) as avg_score
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    WHERE a.age_rating != 'Unknown' AND a.age_rating != 'Неизвестно'
    GROUP BY a.age_rating
    ORDER BY avg_score DESC
    """
)

--- Query: Average Rating by Age Category ---
Insight: Calculates the average user rating for different age ratings to see target audience satisfaction.


,age_rating,avg_score
0,R-17+,6.98
1,R+,6.59
2,PG-13,6.59
3,G,6.13
4,PG,6.02


Query 3: SUM + JOIN + GROUP BY

In [33]:
run_query(
    "Genre Popularity by Views",
    "Sum of site views per genre, calculated through the junction table.",
    """
    SELECT g.name as genre, SUM(s.site_views) as total_views
    FROM genres g
    JOIN anime_genres ag ON g.id = ag.genre_id
    JOIN anime_stats s ON ag.anime_id = s.anime_id
    GROUP BY g.name
    ORDER BY total_views DESC
    LIMIT 10
    """
)


--- Query: Genre Popularity by Views ---
Insight: Sum of site views per genre, calculated through the junction table.


,genre,total_views
0,"['Сёнэн ', ' Приключения ', ' Фэнтези ', ' Экш...",5151
1,"['Приключения ', ' Фэнтези ', ' Экшен ', ' Исэ...",4561
2,"['Приключения ', ' Фэнтези ', ' Экшен ', ' Бое...",3966
3,"['Приключения ', ' Фэнтези ', ' Экшен']",3718
4,"['Комедия ', ' Приключения ', ' Фэнтези ', ' Э...",3123
5,"['Сёнэн ', ' Комедия ', ' Приключения ', ' Фан...",3063
6,['Комедия'],2943
7,"['Сёнэн ', ' Комедия ', ' Экшен ', ' Суперспос...",2772
8,"['Сёнэн ', ' Комедия ', ' Романтика ', ' Повсе...",2348
9,"['Сёнэн ', ' Драма ', ' Комедия ', ' Спорт ', ...",2211


Query 4: WHERE + ORDER BY (Filtering)

In [34]:
run_query(
    "Most Anticipated Upcoming Releases (2026-2027)",
    "Filters future projects and sorts them by user votes/interest.",
    """
    SELECT a.title_ru, a.release_year, s.site_votes, a.status
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    WHERE a.release_year >= 2026
    ORDER BY s.site_votes DESC
    LIMIT 5
    """
)

--- Query: Most Anticipated Upcoming Releases (2026-2027) ---
Insight: Filters future projects and sorts them by user votes/interest.


,title_ru,release_year,site_votes,status
0,Магическая битва: Смертельная миграция,2026,3769,вышел
1,Адский рай 2,2026,3736,вышел
2,Приговорённый быть героем: Тюремные записи 900...,2026,3637,вышел
3,Провожающая в последний путь Фрирен 2,2026,3575,вышел
4,Звёздное дитя 3,2026,2739,вышел


Query 5: Complex JOIN (Studio Performance)

In [35]:
run_query(
    "Studio Reach and Efficiency",
    "Total view count and project count per studio to measure market presence.",
    """
    SELECT st.name as studio, COUNT(ast.anime_id) as project_count, SUM(s.site_views) as total_views
    FROM studios st
    JOIN anime_studios ast ON st.id = ast.studio_id
    JOIN anime_stats s ON ast.anime_id = s.anime_id
    GROUP BY st.name
    HAVING total_views > 0
    ORDER BY total_views DESC
    """
)

--- Query: Studio Reach and Efficiency ---
Insight: Total view count and project count per studio to measure market presence.


,studio,project_count,total_views
0,['J.C.Staff'],330,20619
1,['A-1 Pictures'],187,17476
2,['Madhouse'],266,16259
3,['Bones'],129,14028
4,['TMS Entertainment'],223,12492
...,...,...,...
1416,"['Pb Animation Co. ', ' Ltd. ', ' Paper Plane ...",1,1
1417,['Outline'],1,1
1418,"['Felix Film ', ' Gakyo']",1,1
1419,"['Drive ', ' Massket']",1,1


Query 6: High-Rated Anime

In [36]:
run_query(
    "Top 10 High-Rated Anime (Min 50 Votes)",
    "Finding high-rated anime with very few votes that deserve more attention.",
    """
    SELECT a.title_ru, s.site_rating, s.site_votes
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    WHERE s.site_votes >= 50
    ORDER BY s.site_rating DESC
    LIMIT 10
    """
)

--- Query: Top 10 High-Rated Anime (Min 50 Votes) ---
Insight: Finding high-rated anime with very few votes that deserve more attention.


,title_ru,site_rating,site_votes
0,Re:Zero. Жизнь с нуля в альтернативном мире 4,9.54,618
1,Невероятное приключение ДжоДжо: Гонка «Стально...,9.50,1365
2,Аватар: Легенда об Аанге,9.46,2953
3,Освободите эту ведьму,9.43,1149
4,Доктор Стоун: Научное будущее | Часть 3,9.34,410
5,Ателье колдовских колпаков,9.31,586
6,Вайолет Эвергарден — Фильм,9.29,2127
7,Унесенные призраками,9.26,4232
8,Госпожа Кагуя: в любви как на войне — Лестница...,9.26,775
9,Вистория: Меч и жезл 2,9.22,277


 Query 7: Source Material Influence

In [37]:
run_query(
    "Average Rating by Source Material",
    "Comparing the quality of anime based on their origin (manga, light novel, original, etc.).",
    """
    SELECT source, ROUND(AVG(site_rating), 2) as avg_rating, COUNT(*) as count
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    GROUP BY source
    HAVING count > 5
    ORDER BY avg_rating DESC
    """
)


--- Query: Average Rating by Source Material ---
Insight: Comparing the quality of anime based on their origin (manga, light novel, original, etc.).


,source,avg_rating,count
0,Ранобэ (лайт-новелла),7.07,9
1,Визуальная новелла,6.86,199
2,Манга,6.76,3936
3,Другое,6.63,7
4,Ранобэ,6.62,1087
5,фильм,6.50,6
6,Манхва,6.50,44
7,книга,6.45,98
8,Комикс,6.40,23
9,Мобильная игра,6.31,40


 Query 8: Genre Excellence

In [38]:
run_query(
    "Genres with Highest Average Rating",
    "Top genres by user ratings (minimum 10 titles per genre).",
    """
    SELECT g.name as genre, ROUND(AVG(s.site_rating), 2) as avg_rating
    FROM genres g
    JOIN anime_genres ag ON g.id = ag.genre_id
    JOIN anime_stats s ON ag.anime_id = s.anime_id
    GROUP BY g.name
    HAVING COUNT(ag.anime_id) > 10
    ORDER BY avg_rating DESC
    LIMIT 10
    """
)

--- Query: Genres with Highest Average Rating ---
Insight: Top genres by user ratings (minimum 10 titles per genre).


,genre,avg_rating
0,"['Сёнэн ', ' Комедия ', ' Пародия ', ' Фантаст...",8.40
1,"['Сёнэн ', ' Драма ', ' Комедия ', ' Спорт ', ...",8.16
2,"['Сёнэн ', ' Драма ', ' Комедия ', ' Спорт']",7.82
3,"['Сэйнэн ', ' Фэнтези ', ' Экшен ', ' Сверхъес...",7.62
4,"['Сёнэн ', ' Комедия ', ' Романтика ', ' Повсе...",7.61
5,"['Сёнэн ', ' Комедия ', ' Приключения ', ' Фэн...",7.38
6,"['Сёнэн ', ' Комедия ', ' Повседневность ', ' ...",7.37
7,"['Сэйнэн ', ' Комедия ', ' Повседневность ', '...",7.37
8,"['Сёнэн ', ' Детектив ', ' Комедия ', ' Приклю...",7.31
9,"['Драма ', ' Спорт']",7.21


Query 9: Industry Giants (Most Prolific Studios)

In [39]:
run_query(
    "Top 10 Most Prolific Studios",
    "Studios with the highest number of released projects in the catalog.",
    """
    SELECT st.name as studio, COUNT(ast.anime_id) as project_count
    FROM studios st
    JOIN anime_studios ast ON st.id = ast.studio_id
    GROUP BY st.name
    ORDER BY project_count DESC
    LIMIT 10
    """
)

--- Query: Top 10 Most Prolific Studios ---
Insight: Studios with the highest number of released projects in the catalog.


,studio,project_count
0,['Toei Animation'],480
1,['Unknown'],444
2,['J.C.Staff'],330
3,['Sunrise'],303
4,['Madhouse'],266
5,['Studio Deen'],229
6,['TMS Entertainment'],223
7,['Production I.G'],194
8,['A-1 Pictures'],187
9,['AIC'],130


Query 10: Top 5 Directors by Audience Engagement

In [40]:
run_query(
    "Top 5 Directors by Audience Engagement",
    "Directors whose works collect the most votes on average (minimum 3 projects).",
    """
    SELECT d.name as director, ROUND(AVG(s.site_votes), 0) as avg_votes
    FROM directors d
    JOIN anime_directors ad ON d.id = ad.director_id
    JOIN anime_stats s ON ad.anime_id = s.anime_id
    GROUP BY d.name
    HAVING COUNT(ad.anime_id) >= 3
    ORDER BY avg_votes DESC
    LIMIT 5
    """
)

--- Query: Top 5 Directors by Audience Engagement ---
Insight: Directors whose works collect the most votes on average (minimum 3 projects).


,director,avg_votes
0,['Накадзю Сюнсукэ'],6282.0
1,"['Араки Тэцуро ', ' Коидзука Масаси']",3690.0
2,['Макита Каори'],3547.0
3,"['Цуда Наокацу ', ' Судзуки Кэнъити']",3134.0
4,['Окамото Манабу'],3126.0


Query 11: Yearly Release Activity (Last 15 Years)

In [41]:
run_query(
    "Yearly Release Activity (Last 15 Years)",
    "Dynamics of new anime releases over the last 15 years to analyze industry growth.",
    """
    SELECT release_year, COUNT(*) as release_count
    FROM anime
    WHERE release_year BETWEEN 2009 AND 2024
    GROUP BY release_year
    ORDER BY release_year DESC
    """
)

--- Query: Yearly Release Activity (Last 15 Years) ---
Insight: Dynamics of new anime releases over the last 15 years to analyze industry growth.


,release_year,release_count
0,2024,525
1,2023,538
2,2022,447
3,2021,456
4,2020,370
5,2019,377
6,2018,425
7,2017,383
8,2016,402
9,2015,583


Query 12: Engagement Ratio (Views per Vote)

In [42]:
run_query(
    "Engagement Ratio (Views per Vote)",
    "Analyzing how actively users vote after watching different content formats.",
    """
    SELECT anime_type, ROUND(SUM(site_views)*1.0 / SUM(site_votes), 2) as views_per_vote
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    GROUP BY anime_type
    HAVING SUM(site_votes) > 0
    ORDER BY views_per_vote DESC
    """
)


--- Query: Engagement Ratio (Views per Vote) ---
Insight: Analyzing how actively users vote after watching different content formats.


,anime_type,views_per_vote
0,Неизвестно,0.72
1,ONA,0.38
2,Малометражный сериал,0.25
3,Короткометражный фильм,0.22
4,Сериал,0.18
5,OVA,0.18
6,Спешл,0.17
7,Полнометражный фильм,0.16


Query 13: Quality Hidden Gems (High Rating, Low Views)

In [43]:
run_query(
    "Quality Hidden Gems",
    "High-quality works (rating > 7.5) that remained unnoticed (less than 500 views).",
    """
    SELECT a.title_ru, s.site_rating, s.site_views
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    WHERE s.site_rating > 7.5 AND s.site_views < 500 AND s.site_views > 0
    ORDER BY s.site_rating DESC
    LIMIT 10
    """
)

--- Query: Quality Hidden Gems ---
Insight: High-quality works (rating > 7.5) that remained unnoticed (less than 500 views).


,title_ru,site_rating,site_views
0,Невероятное приключение ДжоДжо: Гонка «Стально...,9.50,1
1,Вайолет Эвергарден — Фильм,9.29,197
2,Унесенные призраками,9.26,278
3,Госпожа Кагуя: в любви как на войне — Лестница...,9.26,221
4,Вистория: Меч и жезл 2,9.22,430
5,Дорохедоро 2,9.22,430
6,Ходячий замок,9.21,449
7,Звёздное дитя 3,9.20,2
8,Ван-Пис,9.17,7
9,Крутой учитель Онидзука,9.14,1


Query 14: Popularity by Age Rating

In [44]:
run_query(
    "Popularity by Age Rating",
    "Total number of views depending on the age rating category.",
    """
    SELECT age_rating, SUM(site_views) as total_views, COUNT(*) as title_count
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    GROUP BY age_rating
    ORDER BY total_views DESC
    """
)

--- Query: Popularity by Age Rating ---
Insight: Total number of views depending on the age rating category.


,age_rating,total_views,title_count
0,PG-13,339138,6364
1,R-17+,112436,1312
2,R+,37045,779
3,G,9760,917
4,Неизвестно,7138,318
5,PG,5631,452


Query 15: Top Dubbing Groups by Average Rating

In [45]:
run_query(
    "Top Dubbing Groups by Average Rating",
    "Dubbing teams whose releases have the highest average rating (minimum 20 projects).",
    """
    SELECT dg.name as group_name, ROUND(AVG(s.site_rating), 2) as avg_rating, COUNT(*) as projects
    FROM dubbing_groups dg
    JOIN anime_dubbing_groups adg ON dg.id = adg.dubbing_group_id
    JOIN anime_stats s ON adg.anime_id = s.anime_id
    GROUP BY dg.name
    HAVING projects > 20
    ORDER BY avg_rating DESC
    LIMIT 10
    """
)

--- Query: Top Dubbing Groups by Average Rating ---
Insight: Dubbing teams whose releases have the highest average rating (minimum 20 projects).


,group_name,avg_rating,projects
0,['MC Entertainment'],6.94,58
1,['AniMaunt'],6.89,30
2,"['AniDUB ', ' SHIZA Project']",6.88,93
3,"['AniDUB ', ' AniLibria']",6.75,33
4,['Helona'],6.69,26
5,"['AniDUB ', ' Persona99']",6.63,50
6,"['AniDUB ', ' AniFilm']",6.59,28
7,['Amazing Dubbing'],6.53,33
8,['AnimeVost'],6.43,23
9,['AniStar'],6.40,97
